In [0]:
%pip install confluent-kafka[avro]
dbutils.library.restartPython()

### Whatever data we have, lets write it to A table to sync checkpoint

In [0]:
import importlib, schema_registry_config, kafka_conn
importlib.reload(schema_registry_config)
importlib.reload(kafka_conn)
 


In [0]:
sr_options =  schema_registry_config.get_kafka_registry_config(dbutils)
TOPIC = kafka_conn.TOPIC_REGISTRY
conf = kafka_conn.get_kafka_conf(dbutils=dbutils)
sr_options
TOPIC

In [0]:
from confluent_kafka import Producer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField 
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer 

### Evolve schema v3 and create events

In [0]:
schema_v2 = """
{
  "type": "record",
  "name": "PostEngagement",
  "namespace": "com.poc",
  "fields": [
    {"name": "post_id", "type": "string"},
    {"name": "type",    "type": "string"},
    {"name": "user_id", "type": "string"},
    {"name": "ts",      "type": "long"},
    {"name": "device",  "type": "string", "default": "unknown"}
  ]
}
"""

In [0]:

sr_client = SchemaRegistryClient(sr_options) 
avro_serializer = AvroSerializer(sr_client, schema_v2, to_dict= lambda obj, ctx: obj)
string_serializer = StringSerializer('utf-8')

In [0]:
events_v2 = [
    {"post_id": "post-3", "type": "view", "user_id": "u20", "ts": 1790900000, "device": "mobile"},
    {"post_id": "post-3", "type": "like", "user_id": "u21", "ts": 1790900001, "device": "web"},
]

In [0]:
 

def delivery_report(err, msg): 

    if err: print("  FAILED:", err) 

    else:   print(f"  delivered key={msg.key().decode()} -> p{msg.partition()} offset {msg.offset()}") 

In [0]:
producer = Producer({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": conf["sasl.username"],
    "sasl.password": conf["sasl.password"],
})

for events in events_v2:
    producer.produce(
        topic=TOPIC,
        key = string_serializer(events["post_id"]),
        value = avro_serializer(events, SerializationContext(TOPIC, MessageField.VALUE)),
        on_delivery = delivery_report
    )
    producer.poll(0)
producer.flush()
print('Done')

In [0]:
v = sr_client.get_latest_version("post-engagement-avro-value")
print("latest version:", v.version, "schema id:", v.schema_id)   # expect version 2

### SS Consumer

In [0]:
sr_op = schema_registry_config.get_spark_avro_sr_options(dbutils)

In [0]:
sr_options = {
   **sr_op,   # auth keys
    "avroSchemaEvolutionMode": "restart"   # <-- ignore schema evolution; keep current shape
}

In [0]:
TOPIC   = kafka_conn.TOPIC_REGISTRY
conf    = kafka_conn.get_kafka_conf(dbutils)
SUBJECT = f"{TOPIC}-value"
SR_ADDR = schema_registry_config.get_sr_address(dbutils)


In [0]:
import kafka_conn, schema_registry_config
from pyspark.sql.functions import col
from pyspark.sql.avro.functions import from_avro


# registry auth + evolution mode = none (the default, but we set it explicitly)


raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load())



In [0]:
sr_options

In [0]:
parsed = raw.select(
    col("partition"), col("offset"),
    from_avro(
        data=col("value"),
        options=sr_options,
        subject=SUBJECT,                  # <-- registry path (handles the 5-byte wire format)
        schemaRegistryAddress=SR_ADDR,
    ).alias("event")
).select("partition", "offset", "event.*")

# display(parsed)


In [0]:
CHK = "s3://retail-data-lake-dev/kafka/_checkpoint/avro_ss_test"
(parsed.writeStream.format("delta").outputMode("append")
    .option("checkpointLocation", CHK)
    .trigger(availableNow=True)
    .toTable("kafka_poc.engagement.avro_mode_restart")).awaitTermination()

display(spark.table("kafka_poc.engagement.avro_mode_restart"))

In [0]:
%sql
drop table kafka_poc.engagement.avro_mode_restart

### Implementation — Strategy 1: Stable contract (pinned schema)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.avro.functions import from_avro 

In [0]:
sr_avro_options = schema_registry_config.get_spark_avro_sr_options(dbutils)   # from_avro-shaped
SR_ADDR         = schema_registry_config.get_sr_address(dbutils)    

In [0]:
# --- PINNED CONTRACT: the exact schema this pipeline depends on (config/JSON) ---
PINNED_SCHEMA = """
{
  "type": "record", "name": "PostEngagement", "namespace": "com.poc",
  "fields": [
    {"name": "post_id", "type": "string"},
    {"name": "type",    "type": "string"},
    {"name": "user_id", "type": "string"},
    {"name": "ts",      "type": "long"}
  ]
}
"""   # <- v1 pinned. Even if producer sends v2, we read only these 4 fields.

In [0]:
raw = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load())

In [0]:
parsed = raw.select ( col("partition"), col("offset"),
                     from_avro(
                         data=col("value"),
                        jsonFormatSchema=PINNED_SCHEMA,     # <-- PIN the reader schema (not subject)
                         options={**sr_avro_options, "mode": "PERMISSIVE"},

                     ).alias('value')
                     )

In [0]:
from pyspark.sql.functions import col, expr
from pyspark.sql.avro.functions import from_avro

In [0]:
parsed = (raw
    # drop the first 5 bytes (magic byte + 4-byte schema ID)
    .withColumn("avro_bytes", expr("substring(value, 6, length(value) - 5)"))
    .select(
        col("partition"), col("offset"),
        from_avro(col("avro_bytes"), PINNED_SCHEMA, {"mode": "PERMISSIVE"}).alias("event")
    ))

In [0]:
parsed.display()